<a href="https://colab.research.google.com/github/Neha609/Machine-Learning-Projects/blob/gh-pages/Ensemble_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# 1. Import required libraries
# ==========================================

import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
# ==========================================
# 2. Create a sample loan-default dataset
# ==========================================

# We are generating data directly so that this notebook
# can run without downloading any external CSV file.

X, y = make_classification(
    n_samples=2000,       # Total customer records
    n_features=8,         # Total input variables
    n_informative=6,      # Useful features for prediction
    n_redundant=2,        # Features related to other features
    weights=[0.75, 0.25], # 75% non-default, 25% default customers
    random_state=42
)

# Convert the generated data into a readable DataFrame
feature_names = [
    "Age",
    "Annual_Income",
    "Loan_Amount",
    "Credit_Score",
    "Debt_To_Income",
    "Employment_Years",
    "Previous_Defaults",
    "Repayment_Score"
]

df = pd.DataFrame(X, columns=feature_names)

# Add the target column
df["Default"] = y

# Display the first five customer records
df.head()

,Age,Annual_Income,Loan_Amount,Credit_Score,Debt_To_Income,Employment_Years,Previous_Defaults,Repayment_Score,Default
0,-0.012075,0.481080,-2.770374,4.363600,-1.168553,1.140204,-0.951355,-1.072968,0
1,-0.468383,-0.719567,-0.430117,-3.409355,1.584517,-2.385020,1.263859,-2.323901,0
2,1.403872,-0.996446,-0.691477,-0.248094,-1.053228,-0.943900,-0.721493,-1.771557,0
3,-0.556372,0.005240,-2.385532,0.316362,-2.153976,-1.154646,-0.855324,-1.065876,0
4,-0.318904,0.220197,-1.006950,-0.598367,-1.285020,-1.051526,-0.833065,-0.800107,0


In [3]:
# ==========================================
# 3. Separate features and target
# ==========================================

# X contains the customer information used for prediction
X = df.drop("Default", axis=1)

# y contains the final output:
# 0 = No Default
# 1 = Default
y = df["Default"]

In [4]:
# ==========================================
# 4. Split data into training and testing sets
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,     # Keep 20% data for final testing
    random_state=42,    # Ensures reproducible results
    stratify=y          # Maintains the same class ratio
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 1600
Testing records: 400


In [5]:
# ==========================================
# 5. Create ensemble and baseline models
# ==========================================

# Baseline model: a single Decision Tree
decision_tree = DecisionTreeClassifier(
    max_depth=6,
    random_state=42
)

# Bagging model:
# Random Forest trains multiple trees independently
# and combines their predictions through majority voting.
random_forest = RandomForestClassifier(
    n_estimators=100,    # Number of decision trees
    max_depth=8,         # Controls tree complexity
    oob_score=True,      # Calculates Out-of-Bag validation score
    random_state=42
)

# Boosting model:
# Each new tree tries to correct the errors made by previous trees.
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,    # Number of sequential trees
    learning_rate=0.1,   # Contribution of each tree
    max_depth=3,         # Shallow trees work as weak learners
    random_state=42
)

# Base models used inside the stacking architecture
base_models = [
    (
        "random_forest",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42
        )
    ),
    (
        "svm",
        SVC(
            probability=True,  # Required for probability-based stacking
            random_state=42
        )
    )
]

# Stacking model:
# Base-model predictions become inputs for Logistic Regression.
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)

In [6]:
# ==========================================
# 6. Train and compare all models
# ==========================================

models = {
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting,
    "Stacking": stacking_model
}

results = []

for model_name, model in models.items():

    # Train the model using training data
    model.fit(X_train, y_train)

    # Predict customer defaults on unseen test data
    predictions = model.predict(X_test)

    # Calculate model accuracy
    accuracy = accuracy_score(y_test, predictions)

    results.append({
        "Model": model_name,
        "Accuracy": round(accuracy, 4)
    })

# Display model comparison
results_df = pd.DataFrame(results).sort_values(
    by="Accuracy",
    ascending=False
)

results_df

,Model,Accuracy
3,Stacking,0.9675
2,Gradient Boosting,0.9450
1,Random Forest,0.9250
0,Decision Tree,0.9125


In [7]:
# ==========================================
# 7. Evaluate the Random Forest in detail
# ==========================================

rf_predictions = random_forest.predict(X_test)

# Confusion matrix shows correct and incorrect predictions
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))

# Classification report shows Precision, Recall and F1-score
print("\nClassification Report:")
print(classification_report(y_test, rf_predictions))

# OOB score evaluates the model using rows not selected
# during bootstrap sampling.
print("OOB Score:", round(random_forest.oob_score_, 4))

Confusion Matrix:
[[295   4]
 [ 26  75]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.95       299
           1       0.95      0.74      0.83       101

    accuracy                           0.93       400
   macro avg       0.93      0.86      0.89       400
weighted avg       0.93      0.93      0.92       400

OOB Score: 0.9337


In [8]:
# ==========================================
# 8. Check important loan-default features
# ==========================================

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
6,Previous_Defaults,0.358452
5,Employment_Years,0.156404
7,Repayment_Score,0.138578
3,Credit_Score,0.085065
0,Age,0.075781
4,Debt_To_Income,0.075077
1,Annual_Income,0.062271
2,Loan_Amount,0.048373


In [9]:
# ==========================================
# 9. Predict default for one new customer
# ==========================================

# Use one test record as a sample new customer
new_customer = X_test.iloc[[0]]

# Predict class
default_prediction = random_forest.predict(new_customer)[0]

# Predict default probability
default_probability = random_forest.predict_proba(new_customer)[0][1]

print("Prediction:", default_prediction)
print("Default Probability:", round(default_probability, 4))

if default_prediction == 1:
    print("Decision: High-risk customer — detailed review required.")
else:
    print("Decision: Lower-risk customer — loan may proceed.")

Prediction: 1
Default Probability: 0.9387
Decision: High-risk customer — detailed review required.


| Model             | Working                                                 |
| ----------------- | ------------------------------------------------------- |
| Decision Tree     | Uses one tree as a baseline                             |
| Random Forest     | Trains multiple trees in parallel and votes             |
| Gradient Boosting | Trains trees sequentially to correct errors             |
| Stacking          | Sends predictions from different models to a meta-model |
